<a href="https://colab.research.google.com/github/theiman112860/Data-Science-Repository/blob/main/GovWatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install lxml beautifulsoup4


In [ ]:
import os
os.environ["GOVINFO_API_KEY"] = "sZy9S9cAdf3ASs3Zt8cXq84bCF43VGenMzLO79Do"


getTitleInfo

In [ ]:
# Colab-ready getTitleInfo() for US Code Titles via GovInfo API Search Service
# - Creates/appends titledata.csv (one row per title per run)
# - Uses POST https://api.govinfo.gov/search?api_key=...
#
# References:
# - GovInfo API Search Service overview and request/response fields
#   https://api.govinfo.gov/search (docs/overview)
# - USCODE help page shows title numbers and search operator usctitlenum
#   and packageId format USCODE-{Year}-title{Title Number}

import os
import json
import time
import datetime as dt
from typing import Dict, Any, Optional, List

import requests
import pandas as pd


GOVINFO_SEARCH_URL = "https://api.govinfo.gov/search"


def _now_iso():
    # ISO8601 with Z for easy logging
    return dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


def _post_search(api_key: str, body: Dict[str, Any], timeout: int = 60) -> Dict[str, Any]:
    """
    GovInfo Search Service:
      POST https://api.govinfo.gov/search?api_key=...
      JSON body: {query, pageSize, offsetMark, sorts, resultLevel?, historical?}
    """
    url = f"{GOVINFO_SEARCH_URL}?api_key={api_key}"
    r = requests.post(url, json=body, timeout=timeout)
    # Helpful error context
    if not r.ok:
        raise RuntimeError(f"GovInfo search failed: HTTP {r.status_code} - {r.text[:500]}")
    return r.json()


def _safe_get(d: Dict[str, Any], path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def getTitleInfo(
    api_key: Optional[str] = None,
    out_csv: str = "titledata.csv",
    title_min: int = 1,
    title_max: int = 54,
    sleep_s: float = 0.1,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Pull 'current' metadata snapshot for each US Code title (1..54) and append to titledata.csv.

    Strategy:
    - Query: collection:(USCODE) and usctitlenum:(<N>)
    - Sort: lastModified DESC, publishdate DESC (most recent first)
    - resultLevel: "package" to avoid granules
    - pageSize: 1

    Returns: DataFrame for this run (title_max-title_min+1 rows, minus failures)
    """
    api_key = api_key or os.environ.get("GOVINFO_API_KEY") or os.environ.get("GPO_API_KEY") or "DEMO_KEY"
    run_id = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    retrieved_at = _now_iso()

    rows = []
    failures = []

    if verbose:
        print(f"Using api_key={'DEMO_KEY' if api_key=='DEMO_KEY' else '***'} | run_id={run_id}")
        print(f"Writing to: {out_csv}")

    for titlenum in range(title_min, title_max + 1):
        # USCODE search operator is documented on the USCODE help page
        # and works via the Search Service query grammar.
        query = f"collection:(USCODE) AND usctitlenum:({titlenum})"

        body = {
            "query": query,
            "pageSize": "1",
            "offsetMark": "*",
            "resultLevel": "package",
            "historical": False,
            "sorts": [
                {"field": "lastModified", "sortOrder": "DESC"},
                {"field": "publishdate", "sortOrder": "DESC"},
            ],
        }

        try:
            data = _post_search(api_key, body)
            results = data.get("results") or []
            if not results:
                # No results for that title (should be rare, but handle it)
                rows.append({
                    "run_id": run_id,
                    "retrieved_at": retrieved_at,
                    "title_num": titlenum,
                    "collectionCode": "USCODE",
                    "packageId": None,
                    "lastModified": None,
                    "dateIssued": None,
                    "dateIngested": None,
                    "resultLink": None,
                    "premisLink": None,
                    "modsLink": None,
                    "zipLink": None,
                    "pdfLink": None,
                    "htmlLink": None,
                    "xmlLink": None,
                    "error": "no_results",
                })
                if verbose:
                    print(f"Title {titlenum:02d}: no results")
                continue

            top = results[0]

            # Links can vary by collection; keep it resilient.
            download = top.get("download") if isinstance(top.get("download"), dict) else {}

            rows.append({
                "run_id": run_id,
                "retrieved_at": retrieved_at,
                "title_num": titlenum,
                "collectionCode": top.get("collectionCode", "USCODE"),
                "packageId": top.get("packageId"),
                "lastModified": top.get("lastModified"),
                "dateIssued": top.get("dateIssued"),
                "dateIngested": top.get("dateIngested"),
                "resultLink": top.get("resultLink"),
                # common download links (may not all exist for USCODE; keep columns anyway)
                "premisLink": download.get("premisLink"),
                "modsLink": download.get("modsLink"),
                "zipLink": download.get("zipLink"),
                "pdfLink": download.get("pdfLink"),
                "htmlLink": download.get("htmlLink"),
                "xmlLink": download.get("xmlLink"),
                "error": "",
            })

            if verbose:
                pkg = top.get("packageId")
                lm  = top.get("lastModified")
                di  = top.get("dateIssued")
                print(f"Title {titlenum:02d}: packageId={pkg} lastModified={lm} dateIssued={di}")

        except Exception as e:
            failures.append((titlenum, str(e)))
            rows.append({
                "run_id": run_id,
                "retrieved_at": retrieved_at,
                "title_num": titlenum,
                "collectionCode": "USCODE",
                "packageId": None,
                "lastModified": None,
                "dateIssued": None,
                "dateIngested": None,
                "resultLink": None,
                "premisLink": None,
                "modsLink": None,
                "zipLink": None,
                "pdfLink": None,
                "htmlLink": None,
                "xmlLink": None,
                "error": str(e)[:500],
            })
            if verbose:
                print(f"Title {titlenum:02d}: ERROR - {str(e)[:160]}")

        # tiny throttle to be polite (and reduce chance of rate-limit headaches)
        if sleep_s:
            time.sleep(sleep_s)

    df_run = pd.DataFrame(rows)

    # Append to CSV (append-only history)
    if os.path.exists(out_csv):
        df_run.to_csv(out_csv, mode="a", index=False, header=False)
    else:
        df_run.to_csv(out_csv, index=False)

    if verbose:
        ok = (df_run["error"].fillna("") == "").sum()
        print(f"\nDone. Rows written this run: {len(df_run)} | OK: {ok} | Errors: {len(df_run)-ok}")
        if failures:
            print("Failures (title_num, error):")
            for t, err in failures[:10]:
                print(f" - {t:02d}: {err[:180]}")

    return df_run


# --- Example usage in Colab ---
# df = getTitleInfo()  # uses env GOVINFO_API_KEY if set, else DEMO_KEY
# df.head()


Run getTitleInfo

In [ ]:
from pathlib import Path

# Ensure target directory exists
Path("/content/sample_data").mkdir(parents=True, exist_ok=True)

# Run bootstrap metadata pull
df_initial = getTitleInfo(
    out_csv="/content/sample_data/titledata.csv",
    verbose=True
)

# Quick sanity check
print(df_initial.shape)
df_initial.head()


/tmp/ipython-input-2101879182.py:72: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
/tmp/ipython-input-2101879182.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


Using api_key=*** | run_id=20260103_195051
Writing to: /content/sample_data/titledata.csv
Title 01: packageId=USCODE-2024-title1 lastModified=2025-06-16T12:43:36Z dateIssued=2024-12-31
Title 02: packageId=USCODE-2024-title2 lastModified=2025-06-16T13:31:45Z dateIssued=2024-12-31
Title 03: packageId=USCODE-2023-title3 lastModified=2025-06-16T13:32:20Z dateIssued=2023-12-31
Title 04: packageId=USCODE-2024-title4 lastModified=2025-06-16T14:30:17Z dateIssued=2024-12-31
Title 05: packageId=USCODE-2024-title5 lastModified=2025-06-16T14:16:45Z dateIssued=2024-12-31
Title 06: packageId=USCODE-2024-title6 lastModified=2025-06-16T14:29:23Z dateIssued=2024-12-31
Title 07: packageId=USCODE-2023-title7 lastModified=2025-08-18T15:01:46Z dateIssued=2023-12-31
Title 08: packageId=USCODE-2024-title8 lastModified=2025-08-18T19:44:32Z dateIssued=2024-12-31
Title 09: packageId=USCODE-2023-title9 lastModified=2025-08-18T19:45:14Z dateIssued=2023-12-31
Title 10: packageId=USCODE-2024-title10 lastModified=20

,run_id,retrieved_at,title_num,collectionCode,packageId,lastModified,dateIssued,dateIngested,resultLink,premisLink,modsLink,zipLink,pdfLink,htmlLink,xmlLink,error
0,20260103_195051,2026-01-03T19:50:51Z,1,USCODE,USCODE-2024-title1,2025-06-16T12:43:36Z,2024-12-31,2025-06-16,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,None,None,
1,20260103_195051,2026-01-03T19:50:51Z,2,USCODE,USCODE-2024-title2,2025-06-16T13:31:45Z,2024-12-31,2025-06-16,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,None,None,
2,20260103_195051,2026-01-03T19:50:51Z,3,USCODE,USCODE-2023-title3,2025-06-16T13:32:20Z,2023-12-31,2024-06-25,https://api.govinfo.gov/packages/USCODE-2023-t...,https://api.govinfo.gov/packages/USCODE-2023-t...,https://api.govinfo.gov/packages/USCODE-2023-t...,https://api.govinfo.gov/packages/USCODE-2023-t...,https://api.govinfo.gov/packages/USCODE-2023-t...,None,None,
3,20260103_195051,2026-01-03T19:50:51Z,4,USCODE,USCODE-2024-title4,2025-06-16T14:30:17Z,2024-12-31,2025-06-16,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,None,None,
4,20260103_195051,2026-01-03T19:50:51Z,5,USCODE,USCODE-2024-title5,2025-06-16T14:16:45Z,2024-12-31,2025-06-16,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,https://api.govinfo.gov/packages/USCODE-2024-t...,None,None,


checkNewData

In [ ]:
import os
import pandas as pd
from typing import Callable, List, Dict, Any, Optional

def checkNewData(
    titledata_csv: str = "/content/sample_data/titledata.csv",
    on_change: Optional[Callable[[int, Dict[str, Any]], Any]] = None,
    use_fallback_dateIssued: bool = True,
    dry_run: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    For each Title, compares the last updated dates for the two most recent rows in titledata.csv.
    If different, triggers on_change(title_num, context) (typically fetchTitle).

    Parameters
    ----------
    titledata_csv : str
        Path to titledata.csv (append-only).
    on_change : callable | None
        Function called when a title changed: on_change(title_num, context_dict).
        If None, does not call anything (still returns changed titles).
    use_fallback_dateIssued : bool
        If lastModified is missing, fall back to dateIssued for comparison.
    dry_run : bool
        If True, do NOT call on_change; just report what would be called.
    verbose : bool
        Print status.

    Returns
    -------
    changed_df : pd.DataFrame
        One row per changed title (includes prev/current timestamps and metadata).
    """
    if not os.path.exists(titledata_csv):
        raise FileNotFoundError(f"Missing {titledata_csv}. Run getTitleInfo() first.")

    df = pd.read_csv(titledata_csv)

    required = {"title_num", "retrieved_at"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"titledata.csv missing required columns: {missing}")

    # Choose comparison field
    compare_col = "lastModified" if "lastModified" in df.columns else None
    if compare_col is None and use_fallback_dateIssued and "dateIssued" in df.columns:
        compare_col = "dateIssued"

    if compare_col is None:
        raise ValueError("titledata.csv has neither lastModified nor dateIssued; cannot compare.")

    # Parse timestamps for ordering. retrieved_at is our ground truth for 'most recent row'
    df["__retrieved_at"] = pd.to_datetime(df["retrieved_at"], errors="coerce")
    # Some govinfo fields may be missing or not parse; keep as string for equality check
    df["__cmp"] = df[compare_col].fillna("").astype(str)

    # Keep only rows with a title_num
    df = df[df["title_num"].notna()].copy()
    df["title_num"] = df["title_num"].astype(int)

    # Sort and pick the two most recent per title
    df = df.sort_values(["title_num", "__retrieved_at"], ascending=[True, True])

    changed_rows = []

    for title_num, g in df.groupby("title_num", sort=True):
        if len(g) < 2:
            # First-run baseline: nothing to compare (treat as unchanged for checkNewData)
            if verbose:
                print(f"Title {title_num:02d}: only one snapshot → skip (baseline)")
            continue

        prev = g.iloc[-2]
        curr = g.iloc[-1]

        prev_cmp = prev["__cmp"]
        curr_cmp = curr["__cmp"]

        if prev_cmp != curr_cmp:
            changed_rows.append({
                "title_num": title_num,
                "compare_field": compare_col,
                "prev_value": prev_cmp,
                "curr_value": curr_cmp,
                "prev_run_id": prev.get("run_id", ""),
                "curr_run_id": curr.get("run_id", ""),
                "prev_retrieved_at": str(prev.get("retrieved_at", "")),
                "curr_retrieved_at": str(curr.get("retrieved_at", "")),
                "curr_packageId": curr.get("packageId", ""),
                "curr_xmlLink": curr.get("xmlLink", ""),
                "curr_zipLink": curr.get("zipLink", ""),
                "curr_resultLink": curr.get("resultLink", ""),
                "curr_error": curr.get("error", ""),
            })

            if verbose:
                print(f"Title {title_num:02d}: CHANGED ({compare_col}: {prev_cmp} → {curr_cmp})")

            # Trigger downstream
            if (on_change is not None) and (not dry_run):
                ctx = {
                    "title_num": title_num,
                    "compare_field": compare_col,
                    "prev": prev.to_dict(),
                    "curr": curr.to_dict(),
                }
                on_change(title_num, ctx)
        else:
            if verbose:
                print(f"Title {title_num:02d}: unchanged")

    changed_df = pd.DataFrame(changed_rows)
    if verbose:
        print(f"\nChanged titles: {len(changed_df)}")

    return changed_df


Run checkNewData

In [ ]:
changed = checkNewData(
    titledata_csv="/content/sample_data/titledata.csv",
    dry_run=True,      # don't download yet
    verbose=True
)

changed.head()


Title 01: only one snapshot → skip (baseline)
Title 02: only one snapshot → skip (baseline)
Title 03: only one snapshot → skip (baseline)
Title 04: only one snapshot → skip (baseline)
Title 05: only one snapshot → skip (baseline)
Title 06: only one snapshot → skip (baseline)
Title 07: only one snapshot → skip (baseline)
Title 08: only one snapshot → skip (baseline)
Title 09: only one snapshot → skip (baseline)
Title 10: only one snapshot → skip (baseline)
Title 11: only one snapshot → skip (baseline)
Title 12: only one snapshot → skip (baseline)
Title 13: only one snapshot → skip (baseline)
Title 14: only one snapshot → skip (baseline)
Title 15: only one snapshot → skip (baseline)
Title 16: only one snapshot → skip (baseline)
Title 17: only one snapshot → skip (baseline)
Title 18: only one snapshot → skip (baseline)
Title 19: only one snapshot → skip (baseline)
Title 20: only one snapshot → skip (baseline)
Title 21: only one snapshot → skip (baseline)
Title 22: only one snapshot → skip

""


fetchtitle

In [ ]:
# ===============================
# govinfo USCODE Title downloader helpers (FIXED)
# - adds missing urlparse/parse_qs imports
# - robust API key injection
# - downloads package ZIP + optional XML/HTML preference handling
# ===============================

import os
import time
import hashlib
import requests
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse  # <-- FIX
from typing import Optional, Dict, Any


def _sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _with_api_key(url: str, api_key: Optional[str]) -> str:
    """
    Append api_key=<key> to a URL querystring if not already present.
    Safe for URLs that already have query parameters.
    """
    if not url:
        return url
    if not api_key:
        return url

    p = urlparse(url)
    q = parse_qs(p.query)

    # do not overwrite if already provided
    if "api_key" in q:
        return url

    q["api_key"] = [api_key]
    new_query = urlencode(q, doseq=True)
    return urlunparse((p.scheme, p.netloc, p.path, p.params, new_query, p.fragment))

import random
import requests
from pathlib import Path
import time

import random, time
import requests
from pathlib import Path

def _download_file(url: str, dest_path: str, retries: int = 6, verbose: bool = True) -> dict:
    dest = Path(dest_path)
    dest.parent.mkdir(parents=True, exist_ok=True)

    last_err = None
    transient = {429, 500, 502, 503, 504}

    for attempt in range(1, retries + 1):
        try:
            if verbose:
                print(f"[download] Attempt {attempt}/{retries}: {url}", flush=True)

            # (connect_timeout, read_timeout)
            with requests.get(url, stream=True, timeout=(15, 30)) as r:
                status = r.status_code

                # retry on transient HTTP
                if status in transient:
                    raise requests.HTTPError(f"HTTP {status}", response=r)

                r.raise_for_status()

                total = 0
                with open(dest, "wb") as f:
                    last_print = time.time()
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if not chunk:
                            continue
                        f.write(chunk)
                        total += len(chunk)

                        # print progress every ~5s
                        if verbose and (time.time() - last_print) > 5:
                            mb = total / (1024 * 1024)
                            print(f"[download] ... {mb:.1f} MB written", flush=True)
                            last_print = time.time()

            sha = _sha256_file(str(dest))
            return {"status": "ok", "bytes": total, "sha256": sha, "dest_path": str(dest)}

        except Exception as e:
            last_err = e
            # cleanup partial
            try:
                if dest.exists():
                    dest.unlink()
            except Exception:
                pass

            # backoff with jitter
            if attempt < retries:
                sleep_s = min(120, (2 ** (attempt - 1)) * (0.8 + 0.6 * random.random()))
                if verbose:
                    print(f"[download] Attempt {attempt} failed: {e}", flush=True)
                    print(f"[download] Sleeping {sleep_s:.1f}s then retry...", flush=True)
                time.sleep(sleep_s)

    return {"status": "fail", "error": str(last_err), "bytes": 0, "sha256": "", "dest_path": ""}


def fetchTitle(
    title_num: int,
    titledata_csv: str,
    prefer: str = "zip",
    out_dir: str = "/content/downloads",
    api_key: Optional[str] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Download a USCODE title package from govinfo, based on packageId stored in titledata_csv.
    - prefer: 'zip' (recommended) or 'xml' (if your pipeline uses XML endpoint)
    - api_key: GOVINFO_API_KEY if required (401 indicates you need it)
    """
    import pandas as pd

    t0 = datetime.now().strftime("%Y%m%d_%H%M%S")
    title_num_int = int(title_num)

    if api_key is None:
        api_key = os.environ.get("GOVINFO_API_KEY") or os.environ.get("GOVINFO_APIKEY")

    df = pd.read_csv(titledata_csv)
    # Expecting columns like: title_num, packageId, download_url (or similar)
    # We'll handle a couple common schemas robustly.
    cand_cols = [c.lower() for c in df.columns]

    # Find title column
    if "title_num" in cand_cols:
        title_col = df.columns[cand_cols.index("title_num")]
    elif "title" in cand_cols:
        title_col = df.columns[cand_cols.index("title")]
    else:
        raise ValueError(f"titledata_csv missing a title column. Found columns: {list(df.columns)}")

    # Find packageId column
    if "packageid" in cand_cols:
        pkg_col = df.columns[cand_cols.index("packageid")]
    elif "package_id" in cand_cols:
        pkg_col = df.columns[cand_cols.index("package_id")]
    else:
        raise ValueError(f"titledata_csv missing packageId column. Found columns: {list(df.columns)}")

    row = df.loc[df[title_col].astype(int) == title_num_int]
    if row.empty:
        raise ValueError(f"Title {title_num_int} not found in {titledata_csv}")

    package_id = str(row.iloc[0][pkg_col])

    # Determine download URL
    if "download_url" in cand_cols:
        dl_col = df.columns[cand_cols.index("download_url")]
        base_url = str(row.iloc[0][dl_col])
    else:
        # Construct govinfo package download URL if not provided
        # ZIP endpoint:
        # https://api.govinfo.gov/packages/<packageId>/zip
        # XML endpoint (often via /xml or /mods etc, depends on what you stored)
        base_url = f"https://api.govinfo.gov/packages/{package_id}/zip"

    prefer = (prefer or "zip").lower().strip()
    if prefer == "xml":
        # If you have a different XML endpoint in your sheet, use it.
        # Otherwise default to the package zip (most reliable).
        url = base_url
        if not url.endswith("/xml"):
            # some users store /zip; if so, keep zip (you can change if you know your endpoint)
            url = base_url
    else:
        url = base_url
        if not url.endswith("/zip"):
            # normalize to zip download if it's a package base URL
            if url.rstrip("/").endswith(package_id):
                url = url.rstrip("/") + "/zip"

    url = _with_api_key(url, api_key)

    dest_dir = Path(out_dir) / f"title{title_num_int}"
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_zip = dest_dir / f"title{title_num_int}_{t0}.zip"

    if verbose:
        print(f"[fetchTitle] Title {title_num_int} download starting...", flush=True)
        print(f"  packageId: {package_id}", flush=True)
        print(f"  URL:  {url}", flush=True)
        print(f"  Dest: {dest_zip}", flush=True)

    dl = _download_file(url, str(dest_zip), retries=3, verbose=verbose)

    result = {
        "status": dl["status"],
        "title_num": title_num_int,
        "packageId": package_id,
        "download_url": url,
        "dest_path": dl.get("dest_path", ""),
        "bytes": dl.get("bytes", 0),
        "sha256": dl.get("sha256", ""),
        "started_at": t0,
        "error": dl.get("error", ""),
    }

    if verbose:
        print(f"[fetchTitle] Done. status={result['status']} bytes={result['bytes']} sha256={result['sha256'][:12]}...", flush=True)

        # --- write a simple manifest row (append-only) ---
    import pandas as pd
    manifest_path = dest_dir / "manifest.csv"
    mf = pd.DataFrame([result])
    if manifest_path.exists():
        mf.to_csv(manifest_path, mode="a", index=False, header=False)
    else:
        mf.to_csv(manifest_path, index=False)


    return result


Run fetchTitle

In [ ]:
# Option 1: set env var (recommended)
import os
#os.environ["GOVINFO_API_KEY"] = "sZy9S9cAdf3ASs3Zt8cXq84bCF43VGenMzLO79Do"

res = fetchTitle(10, titledata_csv="/content/sample_data/titledata.csv", prefer="zip", verbose=True)
res



[fetchTitle] Title 10 download starting...
  packageId: USCODE-2024-title10
  URL:  https://api.govinfo.gov/packages/USCODE-2024-title10/zip?api_key=sZy9S9cAdf3ASs3Zt8cXq84bCF43VGenMzLO79Do
  Dest: /content/downloads/title10/title10_20260103_200905.zip
[download] Attempt 1/3: https://api.govinfo.gov/packages/USCODE-2024-title10/zip?api_key=sZy9S9cAdf3ASs3Zt8cXq84bCF43VGenMzLO79Do
[download] ... 5.0 MB written
[download] ... 10.0 MB written
[download] ... 15.0 MB written
[download] ... 20.0 MB written
[download] ... 25.0 MB written
[download] ... 30.0 MB written
[download] ... 35.0 MB written
[download] ... 40.0 MB written
[download] ... 45.0 MB written
[download] ... 50.0 MB written
[download] ... 55.0 MB written
[download] ... 60.0 MB written
[download] ... 65.0 MB written
[download] ... 70.0 MB written
[download] ... 75.0 MB written
[download] ... 80.0 MB written
[download] ... 85.0 MB written
[download] ... 90.0 MB written
[download] ... 95.0 MB written
[download] ... 100.0 MB writt

{'status': 'ok',
 'title_num': 10,
 'packageId': 'USCODE-2024-title10',
 'download_url': 'https://api.govinfo.gov/packages/USCODE-2024-title10/zip?api_key=sZy9S9cAdf3ASs3Zt8cXq84bCF43VGenMzLO79Do',
 'dest_path': '/content/downloads/title10/title10_20260103_200905.zip',
 'bytes': 714554957,
 'sha256': '4b49e4298482c40f46c7db31c822a34e4063ff03e3f558bb792c7b0178ab89cd',
 'started_at': '20260103_200905',
 'error': ''}

parseTitle

In [ ]:
import re, json, time
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import pandas as pd
from bs4 import BeautifulSoup

def _now_ts():
    return time.strftime("%Y%m%d_%H%M%S")

def _ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)

def _norm_space(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def _write_jsonl(path: str, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def _extract_section_id_from_filename(name: str) -> Optional[str]:
    # examples: ...-sec113.htm , ...-sec1001.htm
    m = re.search(r"-sec(\d+[a-z0-9\-]*)\.htm(l)?$", name.lower())
    return m.group(1) if m else None

def _html_to_text_and_heading(html_path: Path) -> Tuple[str, str]:
    raw = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "lxml")

    for t in soup(["script", "style", "noscript"]):
        t.decompose()

    # Heading
    heading = ""
    h = soup.find(["h1", "h2"])
    if h and h.get_text(strip=True):
        heading = h.get_text(" ", strip=True)
    elif soup.title and soup.title.get_text(strip=True):
        heading = soup.title.get_text(" ", strip=True)

    # Candidate content containers
    candidates = []
    for sel in [
        "article", "main",
        "div#main", "div#content", "div.content", "div#uscode",
        "div#pageContent", "div#body"
    ]:
        node = soup.select_one(sel)
        if node:
            candidates.append(node)

    # Fallback to body
    if not candidates:
        candidates = [soup.body] if soup.body else [soup]

    # pick the container with the most text
    best = max(
        candidates,
        key=lambda n: len(_norm_space(n.get_text(" ", strip=True)))
    )

    text = best.get_text(" ", strip=True)
    heading = _norm_space(heading)
    text = _norm_space(text)

    return heading, text


def parseTitle_html(
    title_num: int,
    unzip_dir: str,
    out_root: str = "/content/outputs",
    max_sections: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Parse GovInfo USCODE Title HTML directory into nodes+relationships CSVs.
    Focus: Section pages (*-sec*.htm).
    """
    ts = _now_ts()
    title_num_int = int(title_num)

    out_dir = Path(out_root) / f"title{title_num_int:02d}"
    _ensure_dir(str(out_dir))

    log_path  = str(out_dir / f"Title{title_num_int:02d}_parse_log_{ts}.jsonl")
    nodes_csv = str(out_dir / f"Title{title_num_int:02d}_nodes_{ts}.csv")
    rels_csv  = str(out_dir / f"Title{title_num_int:02d}_relationships_{ts}.csv")

    base = Path(unzip_dir)
    html_dir = base / "USCODE-2024-title10" / "html"  # default for your current package
    if not html_dir.exists():
        # more robust: find any /html directory under unzip_dir
        candidates = list(base.rglob("html"))
        if not candidates:
            raise FileNotFoundError(f"No html folder found under {unzip_dir}")
        html_dir = candidates[0]

    _write_jsonl(log_path, {"event":"start", "title_num": title_num_int, "html_dir": str(html_dir), "ts": ts})

    # Find section html files
    sec_files = sorted([p for p in html_dir.glob("*.htm*") if re.search(r"-sec\d", p.name.lower())])
    if verbose:
        print(f"[parseTitle_html] Found section HTML files: {len(sec_files)}", flush=True)

    # Write headers
    pd.DataFrame([], columns=["id","labels","properties"]).to_csv(nodes_csv, index=False)
    pd.DataFrame([], columns=["source","type","target","properties"]).to_csv(rels_csv, index=False)

    # Title node
    title_node_id = f"Title:/us/usc/t{title_num_int}"
    title_props = {"number": str(title_num_int), "name": f"Title {title_num_int}", "identifier": f"/us/usc/t{title_num_int}"}
    pd.DataFrame([{
        "id": title_node_id,
        "labels": "Title",
        "properties": json.dumps(title_props, ensure_ascii=False)
    }]).to_csv(nodes_csv, mode="a", index=False, header=False)

    written = 0
    for i, fp in enumerate(sec_files, 1):
        sec_num = _extract_section_id_from_filename(fp.name)
        if not sec_num:
            continue

        heading, text = _html_to_text_and_heading(fp)

        # Build identifiers consistent with your schema style
        identifier = f"/us/usc/t{title_num_int}/s{sec_num}"
        node_id = f"Section:{identifier}"

        props = {
            "identifier": identifier,
            "heading": heading,
            "text": text,
            "title_number": str(title_num_int),
            "source_file": fp.name
        }

        # Append node row
        pd.DataFrame([{
            "id": node_id,
            "labels": "Section",
            "properties": json.dumps(props, ensure_ascii=False)
        }]).to_csv(nodes_csv, mode="a", index=False, header=False)

        # Relationship: Title HAS_SECTION Section
        pd.DataFrame([{
            "source": title_node_id,
            "type": "HAS_SECTION",
            "target": node_id,
            "properties": "{}"
        }]).to_csv(rels_csv, mode="a", index=False, header=False)

        written += 1

        if verbose and written % 200 == 0:
            print(f"[parseTitle_html] Sections written: {written}", flush=True)

        if max_sections is not None and written >= max_sections:
            _write_jsonl(log_path, {"event":"max_sections_reached", "count": written})
            break

    _write_jsonl(log_path, {"event":"done", "sections_written": written, "nodes_csv": nodes_csv, "rels_csv": rels_csv})

    if verbose:
        print(f"[parseTitle_html] Done. sections={written}", flush=True)
        print(f"  nodes: {nodes_csv}", flush=True)
        print(f"  rels : {rels_csv}", flush=True)
        print(f"  log  : {log_path}", flush=True)

    return {
        "title_num": title_num_int,
        "html_dir": str(html_dir),
        "sections_found": len(sec_files),
        "sections_written": written,
        "nodes_csv": nodes_csv,
        "rels_csv": rels_csv,
        "log_path": log_path
    }


Run parseTitle

In [ ]:
parse_html_res = parseTitle_html(
    10,
    unzip_dir=parse_res["unzip_dir"],
    out_root="/content/outputs",
    max_sections=None,
    verbose=True
)
parse_html_res



[parseTitle_html] Found section HTML files: 3847
[parseTitle_html] Sections written: 200
[parseTitle_html] Sections written: 400
[parseTitle_html] Sections written: 600
[parseTitle_html] Sections written: 800
[parseTitle_html] Sections written: 1000
[parseTitle_html] Sections written: 1200
[parseTitle_html] Sections written: 1400
[parseTitle_html] Sections written: 1600
[parseTitle_html] Sections written: 1800
[parseTitle_html] Sections written: 2000
[parseTitle_html] Sections written: 2200
[parseTitle_html] Sections written: 2400
[parseTitle_html] Sections written: 2600
[parseTitle_html] Sections written: 2800
[parseTitle_html] Sections written: 3000
[parseTitle_html] Sections written: 3200
[parseTitle_html] Sections written: 3400
[parseTitle_html] Sections written: 3600
[parseTitle_html] Sections written: 3800
[parseTitle_html] Done. sections=3845
  nodes: /content/outputs/title10/Title10_nodes_20260103_211920.csv
  rels : /content/outputs/title10/Title10_relationships_20260103_21192

{'title_num': 10,
 'html_dir': '/content/downloads/title10/unzipped_20260103_204335/USCODE-2024-title10/html',
 'sections_found': 3847,
 'sections_written': 3845,
 'nodes_csv': '/content/outputs/title10/Title10_nodes_20260103_211920.csv',
 'rels_csv': '/content/outputs/title10/Title10_relationships_20260103_211920.csv',
 'log_path': '/content/outputs/title10/Title10_parse_log_20260103_211920.jsonl'}

Heuristic Code and Analysis

In [ ]:
# ============================================================
# Downstream analysis pipeline for USCODE Title nodes CSV:
# - Heuristics (meaning bullets, risk flags)
# - Business impacts (NAICS hits + effects)
# - Entity linking (agencies / companies / populations)
# - Optional transformer summary (HF)
# - Optional Ollama LLM "two cents"
# - Saves CSV/XLSX/HTML + zip bundle for download
# ============================================================

import os, re, json, time, html
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import pandas as pd
from tqdm import tqdm

# ----------------------------
# 0) Inputs
# ----------------------------
NODES_CSV = parse_html_res["nodes_csv"] if "parse_html_res" in globals() else "/content/outputs/title10/Title10_nodes_*.csv"
OUT_DIR  = "/content/outputs/title10_analysis"
TITLE_NUM = 10

# Controls
RUN_TRANSFORMER = True    # HF summarizer (public model; no token required)
RUN_OLLAMA_LLM   = False  # set True only if Ollama is installed/running
OLLAMA_MODEL     = "llama3.2:1b"  # change if you have others pulled

# For speed/cost control:
MAX_ROWS_TOTAL = None         # None = all rows
MAX_ROWS_LLM   = 200          # how many "interesting" rows to enrich with transformer/LLM
MIN_TEXT_CHARS = 200          # ignore super-short text
SHOW_PROGRESS_EVERY = 200

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def _now_ts():
    return time.strftime("%Y%m%d_%H%M%S")

# ----------------------------
# 1) Load nodes and expand properties
# ----------------------------
def load_nodes(nodes_csv: str) -> pd.DataFrame:
    # Allow glob
    if "*" in nodes_csv:
        import glob
        hits = sorted(glob.glob(nodes_csv))
        if not hits:
            raise FileNotFoundError(f"No files match: {nodes_csv}")
        nodes_csv = hits[-1]

    df = pd.read_csv(nodes_csv)
    # Keep only Section rows (and optionally Title)
    df = df[df["labels"].isin(["Section", "Title"])].copy()

    # Expand JSON properties
    def _parse_props(x):
        try:
            if pd.isna(x): return {}
            return json.loads(x)
        except Exception:
            return {}
    props = df["properties"].apply(_parse_props)
    props_df = pd.json_normalize(props)
    out = pd.concat([df.drop(columns=["properties"]), props_df], axis=1)

    # normalize key columns
    if "identifier" not in out.columns:
        out["identifier"] = None
    if "text" not in out.columns:
        out["text"] = None
    if "heading" not in out.columns:
        out["heading"] = None

    out["text"] = out["text"].fillna("").astype(str)
    out["heading"] = out["heading"].fillna("").astype(str)
    out["identifier"] = out["identifier"].fillna("").astype(str)
    return out

nodes_df = load_nodes(NODES_CSV)
print("Loaded nodes:", nodes_df.shape, "columns:", len(nodes_df.columns))

# focus analysis on Sections only
sections_df = nodes_df[nodes_df["labels"] == "Section"].copy()
if MAX_ROWS_TOTAL:
    sections_df = sections_df.head(MAX_ROWS_TOTAL).copy()

print("Sections for analysis:", len(sections_df))

# ----------------------------
# 2) Heuristics
# ----------------------------

# --- A) risk/meaning heuristics ---
def extract_numbers(text: str) -> List[float]:
    nums = []
    for x in re.findall(r"\$?\b\d[\d,]*(?:\.\d+)?\b", text or ""):
        x = x.replace("$", "").replace(",", "")
        try:
            nums.append(float(x))
        except:
            pass
    return nums

def summarize_single(text: str) -> List[str]:
    """
    Cheap plain-English bullets from keyword cues.
    Keep it deterministic and fast.
    """
    t = (text or "").lower()
    bullets = []

    if "shall" in t and "may" not in t:
        bullets.append("Creates/strengthens an obligation ('shall').")
    if "may" in t and "shall" not in t:
        bullets.append("Allows discretion ('may').")
    if any(k in t for k in ["prohibit", "unlawful", "ban", "may not", "shall not"]):
        bullets.append("Adds or clarifies a prohibition.")
    if any(k in t for k in ["penalty", "fine", "civil penalty", "criminal"]):
        bullets.append("Sets or updates penalties/enforcement.")
    if any(k in t for k in ["audit", "inspect", "investigation", "enforcement"]):
        bullets.append("Touches audits/inspections/enforcement mechanics.")
    if any(k in t for k in ["eligible", "eligibility", "qualify", "entitled"]):
        bullets.append("Defines eligibility/coverage.")
    if any(k in t for k in ["definition", "means", "for purposes of"]):
        bullets.append("Includes a definition affecting interpretation.")
    if any(k in t for k in ["except", "excluding", "exemption", "notwithstanding"]):
        bullets.append("Introduces or modifies an exception/exemption.")

    # Numeric signals
    nums = extract_numbers(text)
    if nums:
        bullets.append(f"Contains numeric thresholds/amounts (e.g., {nums[0]}).")

    if not bullets:
        bullets.append("General statutory text (no strong heuristic flags).")

    return bullets[:5]

def detect_risks(text: str) -> List[Dict[str,str]]:
    t = (text or "").lower()
    risks = []
    def add(kind, why): risks.append({"risk": kind, "rationale": why})

    # Vagueness / discretion
    if any(k in t for k in ["reasonable", "appropriate", "as necessary", "good cause"]):
        add("Ambiguity / discretion", "Contains vague standards that may broaden interpretation.")

    # Fraud/abuse patterns (very rough)
    if any(k in t for k in ["self-attest", "self certify", "certify to", "attestation"]):
        add("Fraud/abuse risk", "Self-attestation language can weaken verification.")

    # Oversight gaps
    if ("audit" in t and "not" in t and "more than" in t) or ("not subject to audit" in t):
        add("Oversight gap", "Audit frequency/coverage appears limited.")

    # Bias / exclusion cues
    if ("citizens" in t) and ("residents" not in t) and ("lawful" not in t):
        add("Equity / exclusion risk", "Citizens-only language may exclude some resident populations.")

    # Privacy/security compliance cues
    if any(k in t for k in ["personal data", "personally identifiable", "pii", "biometric"]):
        add("Privacy compliance", "Touches personal data; may require compliance controls.")
    if any(k in t for k in ["encryption", "multi-factor", "authentication", "security controls"]):
        add("Security compliance", "Introduces security controls that may increase implementation burden.")

    return risks

# --- B) Business mapping (keyword -> NAICS) ---
NAICS_MAP = {
  "data broker": "NAICS 514199 - All Other Information Services",
  "data brokers": "NAICS 514199 - All Other Information Services",
  "data processing": "NAICS 518210 - Data Processing, Hosting, and Related Services",
  "processor": "NAICS 518210 - Data Processing, Hosting, and Related Services",
  "processors": "NAICS 518210 - Data Processing, Hosting, and Related Services",
  "cloud": "NAICS 518210 - Data Processing, Hosting, and Related Services",
  "encryption": "NAICS 541512 - Computer Systems Design Services",
  "multi-factor authentication": "NAICS 541512 - Computer Systems Design Services",
  "mfa": "NAICS 541512 - Computer Systems Design Services",
  "advertising": "NAICS 541810 - Advertising Agencies",
  "targeted advertising": "NAICS 541810 - Advertising Agencies",
  "research": "NAICS 541715 - R&D in Physical, Engineering, and Life Sciences",
  "health": "NAICS 621999 - All Other Misc. Ambulatory Health Care Services",
  "bank": "NAICS 522110 - Commercial Banking",
  "insurance": "NAICS 524210 - Insurance Agencies and Brokerages",
  "telecommunications": "NAICS 517911 - Telecommunications Resellers",
}

def map_affected_entities(text: str) -> List[Dict[str,str]]:
    t = (text or "").lower()
    hits = []
    for k, naics in NAICS_MAP.items():
        if k in t:
            hits.append({"keyword": k, "naics": naics, "impact": "Compliance / operational change"})
    # unique by NAICS
    uniq = {}
    for h in hits:
        uniq.setdefault(h["naics"], h)
    return list(uniq.values())

def infer_business_effects_single(text: str, hits: List[Dict[str,str]]) -> List[str]:
    """
    Heuristic: translate text+hits into short business impact bullets.
    """
    t = (text or "").lower()
    effects = []
    if hits:
        effects.append("May create new compliance obligations for mentioned industries.")
    if any(k in t for k in ["audit", "recordkeeping", "report", "certify", "documentation"]):
        effects.append("May increase compliance, reporting, or recordkeeping costs.")
    if any(k in t for k in ["penalty", "fine", "civil penalty"]):
        effects.append("May increase financial exposure due to penalties/enforcement.")
    if any(k in t for k in ["shall", "must", "required"]):
        effects.append("May require changes to policies, controls, and operational processes.")
    if any(k in t for k in ["prohibit", "ban", "unlawful"]):
        effects.append("May restrict certain business practices or products.")
    return effects[:5] if effects else ["No clear business effect detected (heuristic)."]

# --- C) Lightweight entity linking (regex-based MVP) ---
AGENCY_PAT = re.compile(r"\b(Department of [A-Z][a-zA-Z ]+|Secretary of [A-Z][a-zA-Z ]+|Attorney General|Federal Trade Commission|FTC|FCC|DoD|DHS|HHS|Treasury|IRS)\b")
POP_PAT    = re.compile(r"\b(veterans?|service members?|minors?|children|students?|seniors?|disabled|low-income|residents?|citizens?)\b", re.I)

def entity_linking(text: str) -> Dict[str, List[str]]:
    agencies = sorted(set(m.group(0) for m in AGENCY_PAT.finditer(text or "")))
    pops = sorted(set(m.group(0) for m in POP_PAT.finditer(text or "")))
    # companies: naive pattern for “Inc.” / “LLC” etc.
    companies = sorted(set(re.findall(r"\b[A-Z][A-Za-z0-9&\.\- ]+(?:Inc\.|LLC|Ltd\.|Corporation|Corp\.)\b", text or "")))
    return {"agencies": agencies[:20], "companies": companies[:20], "populations": pops[:20]}

# ----------------------------
# 3) Optional transformer meaning/summary (fast-ish; still heavier than heuristics)
# ----------------------------
_transformer = None

# ----------------------------
# 3) Optional transformer meaning/summary (SAFE + faster)
# ----------------------------
_transformer = None
_tokenizer = None

def _get_transformer():
    global _transformer, _tokenizer
    if _transformer is not None and _tokenizer is not None:
        return _transformer, _tokenizer

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

    model_name = "sshleifer/distilbart-cnn-12-6"
    _tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    _transformer = pipeline(
        "summarization",
        model=model,
        tokenizer=_tokenizer,
        device=-1  # CPU
    )
    return _transformer, _tokenizer

def transformer_summary(text: str, max_input_tokens: int = 900) -> str:
    """
    Summarize with token-level truncation so we never exceed model max length.
    Also uses smaller output lengths for speed.
    """
    if not text or len(text) < 250:
        return ""

    summarizer, tok = _get_transformer()

    # Token-truncate safely (prevents 1094>1024 crash)
    ids = tok.encode(text, truncation=True, max_length=max_input_tokens)
    clipped_text = tok.decode(ids, skip_special_tokens=True)

    # Smaller outputs -> faster on CPU
    out = summarizer(
        clipped_text,
        max_length=80,
        min_length=30,
        do_sample=False,
        truncation=True
    )
    return out[0]["summary_text"].strip()

# ----------------------------
# 4) Optional Ollama LLM "two cents"
# ----------------------------
def _ollama_ok() -> bool:
    import requests
    try:
        r = requests.get("http://127.0.0.1:11434", timeout=2)
        return r.status_code == 200
    except:
        return False

def llm_two_cents(
    section_text: str,
    what_it_means: str,
    risk_flags: str,
    business_naics: str,
    business_effects: str,
    agencies: str,
    populations: str,
    model: str,
) -> str:
    """
    Calls local Ollama /api/generate. Returns short paragraph.
    """
    import requests
    prompt = f"""
You are a cautious policy analyst. Provide a short, practical "two cents" commentary.
- Do NOT hallucinate statutes or facts not in the text.
- If uncertain, say so.
- Focus on: interpretation, who/what may be impacted, unintended consequences, business compliance.

SECTION TEXT:
{section_text[:3500]}

HEURISTIC SUMMARY:
- What it means: {what_it_means}
- Risk flags: {risk_flags or "(none)"}
- Businesses (NAICS): {business_naics or "(none)"}
- Business effects: {business_effects or "(none)"}
- Agencies: {agencies or "(none)"}
- Populations: {populations or "(none)"}

Return 3–6 sentences. No bullets.
""".strip()

    try:
        resp = requests.post(
            "http://127.0.0.1:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=180
        )
        if resp.status_code != 200:
            return f"[LLM error: {resp.text[:200]}]"
        data = resp.json()
        return (data.get("response") or "").strip()
    except Exception as e:
        return f"[LLM error: {e}]"

# ----------------------------
# 5) Choose "interesting subset" for transformer/LLM
# ----------------------------
KEYWORD_INTEREST = re.compile(r"\b(shall|may|must|required|prohibit|unlawful|penalty|fine|audit|except|exemption|notwithstanding|eligib|definition|means)\b", re.I)

def is_interesting(text: str, risks: List[Dict[str,str]], hits: List[Dict[str,str]]) -> bool:
    if len(text) >= 2000:  # long sections often meaningful
        return True
    if risks:
        return True
    if hits:
        return True
    if KEYWORD_INTEREST.search(text or ""):
        return True
    return False

# ----------------------------
# 6) Run analysis
# ----------------------------
records = []
interesting_rows = []

for idx, row in tqdm(list(sections_df.iterrows()), desc="Heuristic analysis (all sections)"):
    text = str(row.get("text", "") or "").strip()
    if len(text) < MIN_TEXT_CHARS:
        continue

    node_id = row.get("id", "")
    identifier = row.get("identifier", "")
    heading = row.get("heading", "")

    bullets = summarize_single(text)
    risks = detect_risks(text)
    hits = map_affected_entities(text)
    ents = entity_linking(text)
    biz_effects = infer_business_effects_single(text, hits)

    rec = {
        "title_number": TITLE_NUM,
        "node_id": node_id,
        "identifier": identifier,
        "heading": heading,
        "text_snippet": text[:240] + ("..." if len(text) > 240 else ""),
        "what_it_means": " | ".join(bullets),
        "risk_flags": " | ".join([f"{r['risk']}: {r['rationale']}" for r in risks]),
        "business_naics": " | ".join(sorted({h["naics"] for h in hits})),
        "business_keywords": " | ".join(sorted({h["keyword"] for h in hits})),
        "business_effects": " | ".join(biz_effects),
        "agencies": ", ".join(ents["agencies"]),
        "companies": ", ".join(ents["companies"]),
        "populations": ", ".join(ents["populations"]),
        "transformer_summary": "",
        "llm_commentary": "",
    }
    records.append(rec)

    if is_interesting(text, risks, hits):
        interesting_rows.append((text, rec))

df_out = pd.DataFrame(records)
print("Heuristic rows:", df_out.shape)

# Build interesting subset DF
df_interest = df_out[
    (df_out["risk_flags"].str.len() > 0) |
    (df_out["business_naics"].str.len() > 0) |
    (df_out["what_it_means"].str.contains("obligation|penalt|eligib|definition|exception", case=False, na=False))
].copy()

# If that heuristic is too strict/loose, we also use the is_interesting() list:
if len(df_interest) < min(200, len(df_out)):
    df_interest = pd.DataFrame([r for _, r in interesting_rows])

df_interest = df_interest.drop_duplicates(subset=["node_id"]).head(MAX_ROWS_LLM).copy()
print("Interesting subset for transformer/LLM:", len(df_interest))

# Save subset immediately (useful for debugging)
SUBSET_CSV = str(Path(OUT_DIR) / f"title{TITLE_NUM:02d}_interesting_subset_{_now_ts()}.csv")
df_interest.to_csv(SUBSET_CSV, index=False)
print("Wrote subset:", SUBSET_CSV)

# ----------------------------
# 7) Optional enrichment on subset only
# ----------------------------
if RUN_TRANSFORMER:
    try:
        _ = _get_transformer()
        for i, r in tqdm(df_interest.iterrows(), total=len(df_interest), desc="Transformer summaries (subset)"):
            node_id = r["node_id"]
            txt = sections_df.loc[sections_df["id"] == node_id, "text"].values
            txt = txt[0] if len(txt) else ""
            df_interest.at[i, "transformer_summary"] = transformer_summary(txt)
    except Exception as e:
        print("Transformer disabled due to error:", e)


if RUN_OLLAMA_LLM:
    if not _ollama_ok():
        print("Ollama not reachable at 127.0.0.1:11434. Set RUN_OLLAMA_LLM=False or start Ollama.")
    else:
        for i, r in tqdm(df_interest.iterrows(), total=len(df_interest), desc="Ollama LLM 'two cents' (subset)"):
            node_id = r["node_id"]
            txt = sections_df.loc[sections_df["id"] == node_id, "text"].values
            txt = txt[0] if len(txt) else ""
            df_interest.at[i, "llm_commentary"] = llm_two_cents(
                section_text=txt,
                what_it_means=r["what_it_means"],
                risk_flags=r["risk_flags"],
                business_naics=r["business_naics"],
                business_effects=r["business_effects"],
                agencies=r["agencies"],
                populations=r["populations"],
                model=OLLAMA_MODEL,
            )

# Merge enriched fields back into df_out
enriched_cols = ["node_id", "transformer_summary", "llm_commentary"]
df_out = df_out.merge(df_interest[enriched_cols], on="node_id", how="left", suffixes=("", "_enr"))
df_out["transformer_summary"] = df_out["transformer_summary_enr"].fillna(df_out["transformer_summary"])
df_out["llm_commentary"] = df_out["llm_commentary_enr"].fillna(df_out["llm_commentary"])
df_out = df_out.drop(columns=["transformer_summary_enr", "llm_commentary_enr"])

# ----------------------------
# 8) Save outputs (CSV/XLSX/HTML) + zip for download
# ----------------------------
ts = _now_ts()
OUT_CSV  = str(Path(OUT_DIR) / f"title{TITLE_NUM:02d}_section_analysis_{ts}.csv")
OUT_XLSX = str(Path(OUT_DIR) / f"title{TITLE_NUM:02d}_section_analysis_{ts}.xlsx")
OUT_HTML = str(Path(OUT_DIR) / f"title{TITLE_NUM:02d}_section_analysis_{ts}.html")

df_out.to_csv(OUT_CSV, index=False)
df_out.to_excel(OUT_XLSX, index=False)

# Simple HTML report
def _esc(x): return html.escape("" if x is None else str(x))

rows_html = []
for _, r in df_out.head(1000).iterrows():  # cap HTML size
    rows_html.append(
        "<tr>"
        f"<td>{_esc(r['identifier'])}</td>"
        f"<td>{_esc(r['heading'])}</td>"
        f"<td>{_esc(r['what_it_means'])}</td>"
        f"<td>{_esc(r['risk_flags'])}</td>"
        f"<td>{_esc(r['business_naics'])}</td>"
        f"<td>{_esc(r['business_effects'])}</td>"
        f"<td>{_esc(r['agencies'])}</td>"
        f"<td>{_esc(r['populations'])}</td>"
        f"<td>{_esc(r.get('transformer_summary',''))}</td>"
        f"<td>{_esc(r.get('llm_commentary',''))}</td>"
        f"</tr>"
    )

html_doc = f"""
<html><head><meta charset="utf-8"/>
<style>
body {{ font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial; padding: 16px; }}
table {{ border-collapse: collapse; width: 100%; font-size: 12.5px; }}
th, td {{ border: 1px solid #ddd; padding: 6px; vertical-align: top; }}
th {{ background: #f6f6f6; text-align: left; position: sticky; top: 0; }}
.small {{ color: #666; }}
</style></head>
<body>
<h2>Title {TITLE_NUM} — Section Analysis</h2>
<p class="small">Rows in CSV/XLSX: {len(df_out)}. HTML shows first 1000 rows.</p>
<table>
<thead>
<tr>
<th>Identifier</th><th>Heading</th><th>What it means</th><th>Risk flags</th>
<th>Business NAICS</th><th>Business effects</th><th>Agencies</th><th>Populations</th>
<th>Transformer summary</th><th>LLM commentary</th>
</tr>
</thead>
<tbody>
{''.join(rows_html)}
</tbody>
</table>
</body></html>
""".strip()

Path(OUT_HTML).write_text(html_doc, encoding="utf-8")

print("Wrote:")
print(" ", OUT_CSV)
print(" ", OUT_XLSX)
print(" ", OUT_HTML)

# Zip bundle
ZIP_PATH = str(Path(OUT_DIR) / f"title{TITLE_NUM:02d}_analysis_bundle_{ts}.zip")
import zipfile
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT_CSV,  arcname=Path(OUT_CSV).name)
    z.write(OUT_XLSX, arcname=Path(OUT_XLSX).name)
    z.write(OUT_HTML, arcname=Path(OUT_HTML).name)
    z.write(SUBSET_CSV, arcname=Path(SUBSET_CSV).name)

print("Bundle:", ZIP_PATH)

# Download to laptop (Colab)
from google.colab import files
files.download(ZIP_PATH)


Loaded nodes: (3846, 9) columns: 9
Sections for analysis: 3845


Heuristic analysis (all sections): 100%|██████████| 3845/3845 [00:16<00:00, 237.95it/s]


Heuristic rows: (3845, 15)
Interesting subset for transformer/LLM: 200
Wrote subset: /content/outputs/title10_analysis/title10_interesting_subset_20260103_214052.csv


Device set to use cpu
Transformer summaries (subset): 100%|██████████| 200/200 [1:07:53<00:00, 20.37s/it]


Wrote:
  /content/outputs/title10_analysis/title10_section_analysis_20260103_224849.csv
  /content/outputs/title10_analysis/title10_section_analysis_20260103_224849.xlsx
  /content/outputs/title10_analysis/title10_section_analysis_20260103_224849.html
Bundle: /content/outputs/title10_analysis/title10_analysis_bundle_20260103_224849.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>